# SNN Step 3 Sweep — Colab Runner
**Mixed ops (all 8), h∈{20,50}, K=[1..6], 3 conditions, 300 epochs**

Steps:
1. Check GPU
2. Mount Google Drive (断连保护)
3. Clone repo & install deps
4. Smoke test
5. Full sweep (结果实时同步到 Drive)
6. Preview summary
7. 下载结果

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/SNN_step3_runs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Results will sync to: {DRIVE_DIR}')

In [ ]:
# 3. Clone repo
!git clone https://github.com/KunpengXu-04/SNN-async-delays.git
%cd SNN-async-delays/snn_async_delays

# Symlink runs/ -> Drive, so every completed run is saved immediately
import os
os.makedirs('/content/drive/MyDrive/SNN_step3_runs', exist_ok=True)
if not os.path.exists('runs'):
    os.symlink('/content/drive/MyDrive/SNN_step3_runs', 'runs')
else:
    # runs/ already exists from a previous session — reuse it
    import shutil
    shutil.copytree('runs', '/content/drive/MyDrive/SNN_step3_runs', dirs_exist_ok=True)
    shutil.rmtree('runs')
    os.symlink('/content/drive/MyDrive/SNN_step3_runs', 'runs')

print('runs/ -> Google Drive symlink created.')
print('Existing runs on Drive:', os.listdir('runs'))

In [ ]:
# 4. Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install pyyaml matplotlib numpy -q
print('Dependencies installed.')

In [ ]:
# 5. Smoke test (K=1, 10 epochs) — should finish in ~60s
!python -m scripts.run_step3 \
    --config configs/step3_multiquery_multiop.yaml \
    --K 1 --epochs 10 --device cuda

In [ ]:
# Verify smoke test output exists, then re-run to confirm skip logic
import os, glob
smoke_dirs = glob.glob('runs/step3_mixedops_*_K1_seed42')
for d in smoke_dirs:
    path = os.path.join(d, 'eval_results.json')
    print(f'{d}: eval_results.json exists = {os.path.exists(path)}')

print('\nRe-running smoke test — should print [SKIPPED]:')
!python -m scripts.run_step3 \
    --config configs/step3_multiquery_multiop.yaml \
    --K 1 --epochs 10 --device cuda

In [ ]:
# 6. Full sweep — 3 conditions x 2 hidden sizes x 6 K values = 36 runs
# Results are written directly to Google Drive after each run.
# If the session disconnects, re-run cells 1-4 to reconnect,
# then re-run this cell — already-completed runs will be skipped
# automatically (eval_results.json already exists on Drive).
!python -m scripts.run_step3 \
    --config configs/step3_multiquery_multiop.yaml \
    --sweep --device cuda

In [ ]:
# 7. Preview summary
import json
with open('runs/step3_sweep_summary.json') as f:
    summary = json.load(f)

print(f'Conditions in summary: {list(summary.keys())}\n')
for cname, data in summary.items():
    mk = data['max_K_by_tau']
    print(f'{cname}:')
    print(f'  maxK@0.95={mk.get("0.95")}, maxK@0.90={mk.get("0.90")}, maxK@0.85={mk.get("0.85")}')

In [ ]:
# 8. Package results and save to Drive
import shutil, os, glob

items_to_zip = []
if os.path.exists('runs/step3_sweep_summary.json'):
    items_to_zip.append('runs/step3_sweep_summary.json')
if os.path.exists('runs/step3_plots'):
    items_to_zip.append('runs/step3_plots')
for d in os.listdir('runs'):
    if d.startswith('step3_') and os.path.isdir(f'runs/{d}'):
        items_to_zip.append(f'runs/{d}')

os.makedirs('export', exist_ok=True)
for item in items_to_zip:
    if os.path.exists(item):
        dest = f'export/{item}'
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if os.path.isdir(item):
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)

shutil.make_archive('step3_results', 'zip', 'export')
print('step3_results.zip ready.')

# Also copy zip to Drive as backup
shutil.copy2('step3_results.zip', '/content/drive/MyDrive/SNN_step3_runs/step3_results.zip')
print('Zip also saved to Google Drive.')

In [ ]:
# 9. Download zip directly to your computer
from google.colab import files
files.download('step3_results.zip')